<a href="https://colab.research.google.com/github/enzoalvino00/Analitica-Empresarial-Integrada/blob/main/Unidad%20_3_Analitiica_Predictiva_y_Prescriptiva/Semana_09_Empresa_analitica6/Anal%C3%ADtica_Empresarial_Integrada_Lab11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# TECSUP 2026-II - ANALÍTICA EMPRESARIAL INTEGRADA

## LABORATORIO DIRIGIDO N.° 11: Optimización de inventario

| | |
|---|---|
| **Docente** | Pilar Rocío Sayán Mejía |
| **Laboratorio** | Optimización de inventario |
| **N.° de laboratorio** | 11 |
| **Tema** | Optimización de inventario mediante programación entera y análisis de sensibilidad |
| **Plan curricular** | Big Data y Ciencia de Datos |
| **Código** | C28 |
| **Periodo académico** | 2026-II |
| **Duración** | Por confirmar según la programación de la sesión |
| **Modalidad** | Por confirmar según las indicaciones de la docente |

---

**Estudiante:** Enzo Gabriel Alvino Carhuaricra  
**Sección:** 6C28  
**Fecha:** 28/10/2026

### **Pregunta de negocio**
**¿Cómo debe distribuirse una capacidad limitada de inventario entre productos para maximizar el valor potencial de venta sin descuidar niveles mínimos de servicio?**

> El objetivo no es solo obtener `Optimal`. Debe demostrarse cuánto mejora la decisión frente a una regla intuitiva.

In [1]:
# ═════════════════════════
# INSTALACIÓN DE LIBRERÍAS
# ═════════════════════════

# Instala las librerías necesarias en Google Colab; PuLP mantiene una versión fija para reproducir los resultados.
%pip install -q "ucimlrepo>=0.0.7" "pulp==3.3.0" plotly openpyxl

In [2]:
# ══════════════════════════════════════════════════════════════
# LIBRERÍAS Y CONFIGURACIÓN DEL ENTORNO
# ══════════════════════════════════════════════════════════════

# Librerías para análisis de datos, optimización y visualización
## pandas y numpy procesan los datos; PuLP optimiza y Plotly genera gráficos
import sys
import numpy as np  ### Permite realizar operaciones numéricas y cálculos con arreglos
import pandas as pd  ### Facilita la manipulación y el análisis de datos tabulares
import plotly.express as px  ### Permite crear gráficos interactivos
import pulp as pl  ### Proporciona herramientas para formular y resolver modelos de optimización
from ucimlrepo import fetch_ucirepo  ### Permite descargar conjuntos de datos del repositorio UCI

# Registro de versiones para reproducibilidad
## Permite identificar las versiones utilizadas al ejecutar el laboratorio
print("Python:", sys.version.split()[0], "· pandas:", pd.__version__, "· PuLP:", pl.__version__)  ### Muestra las versiones instaladas de Python, pandas y PuLP para reproducir el entorno

# Configuración de visualización
## Permite mostrar hasta 50 columnas al imprimir tablas
pd.set_option("display.max_columns", 50)  ### Establece el máximo de columnas visibles en las tablas de pandas

Python: 3.13.18 · pandas: 2.2.3 · PuLP: 3.3.0


# **1. Descarga de datos reales y trazabilidad**

In [3]:
# ══════════════════════════════════════════════════════════════
# CARGA DEL DATASET ONLINE RETAIL II
# ══════════════════════════════════════════════════════════════

# Carga de datos desde UCI con alternativas de respaldo
## Importación de herramientas para descargar, abrir y gestionar archivos
from io import BytesIO  ### Permite trabajar con archivos binarios en memoria
from urllib.request import urlopen  ### Permite descargar archivos desde una URL
from zipfile import ZipFile  ### Permite abrir archivos comprimidos ZIP
from pathlib import Path  ### Permite gestionar rutas y comprobar archivos locales

# Acceso mediante la API de UCI
try:
    retail = fetch_ucirepo(id=502)  ### Obtiene Online Retail II mediante el identificador 502
    df = retail.data.original.copy()  ### Copia los datos originales en un DataFrame
    fuente = "API ucimlrepo, UCI ID 502"  ### Registra la fuente utilizada
except Exception as exc:
    print("API no disponible:", type(exc).__name__)  ### Muestra el tipo de error si falla la API

    # Descarga del ZIP oficial como alternativa
    ## Se utiliza esta opción cuando no es posible acceder mediante la API
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"  ### Define la URL oficial del archivo ZIP
        with urlopen(url, timeout=120) as response:  ### Descarga el archivo con un máximo de 120 segundos de espera
            with ZipFile(BytesIO(response.read())) as z:  ### Abre el ZIP directamente desde la memoria
                name = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))  ### Identifica el archivo Excel dentro del ZIP
                df = pd.concat(pd.read_excel(BytesIO(z.read(name)), sheet_name=None).values(), ignore_index=True)  ### Combina todas las hojas del Excel en un DataFrame
        fuente = "ZIP oficial UCI, ID 502"  ### Registra la fuente alternativa
    except Exception:
        # Carga del archivo local como último recurso
        ## Se utiliza un archivo descargado previamente si fallan las dos opciones anteriores
        local = Path("online_retail_II.xlsx")  ### Define la ruta del archivo Excel local
        if not local.exists():
            raise FileNotFoundError("Descargue online_retail_II.xlsx desde UCI y súbalo a Colab.")  ### Solicita el archivo si no se encuentra
        df = pd.concat(pd.read_excel(local, sheet_name=None).values(), ignore_index=True)  ### Lee y combina todas las hojas del archivo local
        fuente = "Archivo local descargado de UCI"  ### Registra la fuente local

# Inspección inicial de los datos
## Se verifica el origen, el tamaño y la estructura del dataset
print("Fuente:", fuente, "· ejecución UTC:", pd.Timestamp.now(tz="UTC").isoformat())  ### Muestra la fuente y la fecha y hora de ejecución
print("Dimensión:", df.shape, "· columnas:", df.columns.tolist())  ### Muestra las dimensiones y los nombres de las columnas
display(df.head())  ### Visualiza las primeras cinco filas para inspeccionar los datos

API no disponible: DatasetNotFoundError
Fuente: ZIP oficial UCI, ID 502 · ejecución UTC: 2026-10-29T03:49:04.627271+00:00
Dimensión: (1067371, 8) · columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### **Verificación de la carga del dataset Online Retail II**

El dataset se cargó correctamente mediante el archivo ZIP oficial de UCI, ya que la API no encontró el conjunto de datos solicitado. La alternativa de descarga funcionó y permitió obtener **1,067,371 registros y 8 columnas**. La vista previa muestra información sobre facturas, productos, cantidades vendidas, fechas, precios, clientes y países. Por tanto, los datos están disponibles para continuar con el análisis y la preparación del modelo de optimización de inventario.

| Indicador | Valor obtenido | Significado |
|---|---|---|
| Fuente de datos | ZIP oficial de UCI, ID 502 | Se utilizó el método alternativo de descarga. |
| Registros | 1,067,371 | Cantidad de filas disponibles para el análisis. |
| Columnas | 8 | Variables relacionadas con ventas y clientes. |
| Variables relevantes | `StockCode`, `Quantity`, `Price`, `InvoiceDate` | Permiten identificar productos, cantidades vendidas, precios y fechas de venta. |
| API de UCI | No disponible | La API presentó `DatasetNotFoundError`, pero la descarga alternativa fue exitosa. |
| Estado de carga | Correcto | El DataFrame está listo para las siguientes etapas del laboratorio. |

## **Normalización de nombres**

In [4]:
# ══════════════════════════════════════════════════════════════
# NORMALIZACIÓN DE NOMBRES DE COLUMNAS
# ══════════════════════════════════════════════════════════════

# Identificación de columnas según la fuente de datos
## Se contemplan nombres alternativos para mantener la compatibilidad entre fuentes
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_",""): c for c in columns}  ### Normaliza los nombres para facilitar la comparación
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_","")  ### Normaliza cada nombre candidato
        if key in normalized:
            return normalized[key]  ### Devuelve el nombre original de la columna encontrada
    return None  ### Devuelve None si no existe ninguna coincidencia

# Correspondencia entre nombres originales y nombres estandarizados
## Se unifican las columnas para utilizar los mismos nombres en el resto del notebook
mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),  ### Identifica el número de factura
    "StockCode": find_col(df.columns, ["StockCode"]),  ### Identifica el código del producto
    "Description": find_col(df.columns, ["Description"]),  ### Identifica la descripción del producto
    "Quantity": find_col(df.columns, ["Quantity"]),  ### Identifica la cantidad vendida
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),  ### Identifica la fecha de facturación
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),  ### Identifica el precio unitario
    "Country": find_col(df.columns, ["Country"]),  ### Identifica el país de la transacción
}

# Validación de columnas obligatorias
## Se comprueba que existan las variables necesarias para continuar con el análisis
required = ["InvoiceNo","StockCode","Quantity","InvoiceDate","UnitPrice","Country"]  ### Define las columnas indispensables
missing = [k for k in required if mapping[k] is None]  ### Identifica las columnas que no se encontraron
if missing:
    raise ValueError(f"No se identificaron columnas obligatorias: {missing}")  ### Detiene la ejecución si falta alguna columna

# Estandarización de nombres y verificación
## Se renombran las columnas encontradas y se muestra la correspondencia utilizada
df = df.rename(columns={v:k for k,v in mapping.items() if v is not None})  ### Aplica los nombres estandarizados sin modificar los datos
print(mapping)  ### Muestra la correspondencia entre los nombres originales y los estandarizados

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'Country': 'Country'}


### **Estandarización de nombres de columnas**

La correspondencia confirma que el dataset utiliza los nombres `Invoice` y `Price` en lugar de `InvoiceNo` y `UnitPrice`. La función identificó correctamente estas diferencias y las columnas obligatorias están disponibles. Como el código ejecutó sin generar un error, se puede continuar con la preparación de los datos para el modelo de optimización de inventario.

| Columna estandarizada | Columna original | Estado |
|---|---|---|
| `InvoiceNo` | `Invoice` | Identificada |
| `StockCode` | `StockCode` | Identificada |
| `Description` | `Description` | Identificada |
| `Quantity` | `Quantity` | Identificada |
| `InvoiceDate` | `InvoiceDate` | Identificada |
| `UnitPrice` | `Price` | Identificada |
| `Country` | `Country` | Identificada |

# **2. Control de calidad**

In [5]:
# ══════════════════════════════════════════════════════════════
# LIMPIEZA Y FILTRADO DE LOS DATOS DE VENTAS
# ══════════════════════════════════════════════════════════════

# Conversión de tipos de datos
## Se convierten fecha, cantidad y precio a sus tipos correspondientes
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")  ### Convierte las fechas no válidas en valores vacíos
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")  ### Convierte las cantidades a valores numéricos
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")  ### Convierte los precios unitarios a valores numéricos

# Limpieza del código de producto
## Se eliminan los registros sin código y se estandariza el formato
df = df[df["StockCode"].notna()].copy()  ### Conserva registros con código de producto disponible
df["StockCode"] = df["StockCode"].astype(str).str.strip()  ### Convierte los códigos a texto y elimina espacios al inicio y al final

# Identificación de facturas canceladas
## Las facturas que empiezan con C corresponden a cancelaciones y no se consideran ventas válidas
cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")  ### Identifica las facturas canceladas

# Filtrado de ventas válidas
## Se conservan registros con datos completos y valores positivos de cantidad y precio
clean = df.loc[(~cancel) & df["InvoiceDate"].notna() &
               df["StockCode"].ne("") & df["Country"].notna() &
               (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()  ### Excluye cancelaciones y registros que incumplen las condiciones de limpieza

# Selección del mercado principal
## Se filtran las ventas correspondientes al Reino Unido
uk = clean[clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")].copy()  ### Conserva únicamente las ventas del Reino Unido

# Resumen de la limpieza
## Se verifican las cancelaciones identificadas y el periodo de ventas disponible
print("Cancelaciones:", int(cancel.sum()), "· filas válidas UK:", len(uk))  ### Muestra el total de cancelaciones y las filas válidas del Reino Unido
print("Periodo:", uk["InvoiceDate"].min(), "→", uk["InvoiceDate"].max())  ### Muestra las fechas inicial y final del periodo analizado

Cancelaciones: 19494 · filas válidas UK: 958501
Periodo: 2009-12-01 07:45:00 → 2011-12-09 12:49:00


### **Limpieza y selección de ventas del Reino Unido**

El proceso identificó **19,494 facturas de cancelación** y conservó **958,501 registros de ventas válidas del Reino Unido**. Los datos abarcan desde el 1 de diciembre de 2009 hasta el 9 de diciembre de 2011. Se excluyeron las cancelaciones, los registros con información incompleta y las transacciones con cantidades o precios no positivos, obteniendo una base depurada para analizar la demanda por producto y desarrollar el modelo de optimización de inventario.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Cancelaciones identificadas | 19,494 | Facturas cuyo código comienza con `C`. |
| Filas válidas del Reino Unido | 958,501 | Registros que cumplen los criterios de limpieza y filtrado. |
| Fecha inicial | 01/12/2009 | Inicio del periodo de análisis. |
| Fecha final | 09/12/2011 | Fin del periodo de análisis. |
| Duración del periodo | Aproximadamente 2 años | Horizonte histórico disponible para estudiar las ventas. |

### **Pregunta 1**
**¿Por qué es incorrecto usar directamente la suma de Quantity sin revisar cancelaciones y precios?**

Usar directamente la suma de Quantity es incorrecto porque el dataset contiene cancelaciones, devoluciones y transacciones con precios no válidos. Las cancelaciones pueden generar cantidades negativas que distorsionan la demanda real, mientras que los precios nulos, cero o negativos pueden afectar el cálculo del valor económico de las ventas. Por ello, es necesario limpiar los datos y conservar las ventas válidas antes de estimar la demanda y optimizar el inventario, evitando resultados sesgados en la asignación de recursos.

# **3. Construcción de parámetros reales**

In [6]:
# ══════════════════════════════════════════════════════════════
# SELECCIÓN DE LOS 10 PRODUCTOS PRINCIPALES Y ESTIMACIÓN DE DEMANDA
# ══════════════════════════════════════════════════════════════

# Definición de las últimas 12 semanas completas
## Se excluye la semana parcial porque el dataset termina el viernes 9 de diciembre de 2011
last_obs = uk["InvoiceDate"].max().normalize()  ### Obtiene la última fecha disponible sin considerar la hora
last_complete_sunday = last_obs - pd.Timedelta(days=last_obs.weekday() + 1)  ### Calcula el domingo de la última semana completa
first_monday = last_complete_sunday - pd.Timedelta(weeks=12) + pd.Timedelta(days=1)  ### Calcula el lunes inicial de la ventana de 12 semanas

# Filtrado del periodo de análisis
## Se conservan únicamente las ventas registradas dentro de las 12 semanas completas
recent = uk.loc[
    (uk["InvoiceDate"] >= first_monday) &
    (uk["InvoiceDate"] < last_complete_sunday + pd.Timedelta(days=1))
].copy()  ### Filtra las transacciones entre las fechas establecidas
recent["week"] = recent["InvoiceDate"].dt.to_period("W-SUN").dt.start_time  ### Asigna a cada transacción el lunes de su semana
assert recent["week"].nunique() == 12, "Revisar cobertura de las 12 semanas."  ### Verifica que existan exactamente 12 semanas

# Agregación semanal por producto
## Se calcula la cantidad vendida y el precio mediano de cada producto por semana
weekly = recent.groupby(["StockCode", "week"], as_index=False).agg(
    units=("Quantity", "sum"),  ### Suma las unidades vendidas por producto y semana
    price=("UnitPrice", "median")  ### Calcula el precio mediano semanal
)

# Cálculo de parámetros de demanda por producto
## Se resume la frecuencia de venta y la demanda total observada en el periodo
sku_params = weekly.groupby("StockCode").agg(
    weeks_present=("week", "nunique"),  ### Cuenta las semanas en las que aparece cada producto
    demand_total=("units", "sum")  ### Calcula las unidades acumuladas por producto
).reset_index()
sku_params["weekly_demand"] = sku_params["demand_total"] / 12  ### Estima la demanda semanal promedio del periodo
sku_params["price"] = sku_params["StockCode"].map(
    recent.groupby("StockCode")["UnitPrice"].median()  ### Obtiene el precio mediano histórico de cada producto
)

# Selección de productos con suficiente presencia semanal
## Se priorizan productos con ventas en al menos 8 semanas; si no son suficientes, se reduce el umbral
eligible = sku_params.loc[sku_params["weeks_present"] >= 8].copy()  ### Filtra los productos con presencia en 8 o más semanas
if len(eligible) < 10:
    eligible = sku_params.loc[sku_params["weeks_present"] >= 6].copy()  ### Reduce el mínimo a 6 semanas si faltan productos elegibles
if len(eligible) < 10:
    raise ValueError("Menos de 10 SKU elegibles; revisar datos y regla de selección.")  ### Detiene el proceso si no hay suficientes productos

# Selección de los 10 productos principales
## Se ordenan por demanda total descendente y por código como criterio de desempate
top = eligible.sort_values(["demand_total", "StockCode"], ascending=[False, True]).head(10).copy()  ### Selecciona los 10 productos de mayor demanda acumulada
top["demanda"] = np.maximum(np.ceil(top["weekly_demand"]), 1).astype(int)  ### Redondea hacia arriba la demanda semanal y garantiza un mínimo de una unidad

# Construcción de la tabla de parámetros para la optimización
## Se organizan los datos que utilizará el modelo matemático
parametros = top.set_index("StockCode")[["demanda", "price", "weeks_present", "demand_total"]].copy()  ### Conserva la demanda, el precio y la presencia semanal de cada producto

# Verificación de la selección
## Se muestran las fechas analizadas, los parámetros de los productos y la demanda estimada total
print("Ventana:", first_monday.date(), "a", last_complete_sunday.date())  ### Muestra el periodo de 12 semanas completas
display(parametros)  ### Presenta los parámetros de los 10 productos seleccionados
print("Demanda esperada total:", int(parametros["demanda"].sum()))  ### Muestra la demanda semanal estimada conjunta de los 10 productos

Ventana: 2011-09-12 a 2011-12-04


,demanda,price,weeks_present,demand_total
StockCode,,,,
22197,1809,0.85,12,21703
84077,1223,0.29,12,14676
85099B,1175,2.08,12,14098
22086,1080,2.95,12,12951
23084,959,2.08,12,11505
84879,860,1.69,12,10310
22952,765,0.55,12,9177
22577,746,0.29,12,8948
22578,693,0.29,12,8305


Demanda esperada total: 9982


### **Estimación de la demanda semanal de los 10 productos principales**

Se analizaron las 12 semanas completas comprendidas entre el 12 de septiembre y el 4 de diciembre de 2011. Se seleccionaron los 10 productos con mayor demanda acumulada que registraron ventas en las 12 semanas. La demanda semanal estimada conjunta es de **9,982 unidades**, mientras que los precios medianos permiten representar el valor económico de cada producto. Estos parámetros servirán como entrada para el modelo matemático de optimización de inventario.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Periodo analizado | 12/09/2011 al 04/12/2011 | Ventana de 12 semanas completas. |
| Productos seleccionados | 10 SKU | Productos con mayor demanda acumulada y presencia en las 12 semanas. |
| Demanda semanal estimada total | 9,982 unidades | Suma de las demandas semanales estimadas de los 10 productos. |
| Producto de mayor demanda | SKU 22197 | Demanda estimada de 1,809 unidades por semana. |
| Producto de menor demanda entre los seleccionados | SKU 21915 | Demanda estimada de 672 unidades por semana. |
| Precio mediano | Entre 0.29 y 2.95 | Referencia del precio unitario por producto, expresada en la moneda del dataset. |

### **Pregunta 2**
**¿Qué significa una semana sin ventas observadas para un SKU y por qué no prueba demanda cero?**

Una semana sin ventas observadas para un SKU significa que no se registraron transacciones de ese producto durante esa semana. Sin embargo, esto no demuestra que la demanda haya sido cero, porque el producto pudo tener poca rotación, falta de inventario, problemas de disponibilidad o simplemente ningún cliente realizó una compra durante ese periodo. Por ello, la ausencia de ventas debe interpretarse como falta de observaciones de compra, no necesariamente como ausencia de demanda real.

# 4. Supuestos explícitos del escenario

In [7]:
# ══════════════════════════════════════════════════════════════
# DEFINICIÓN DE RESTRICCIONES DE CAPACIDAD Y SERVICIO
# ══════════════════════════════════════════════════════════════

# Configuración de los supuestos académicos del caso
## Se establecen los porcentajes de capacidad, servicio mínimo y concentración máxima por producto
CAPACITY_RATIO = 0.70  ### La capacidad total equivale al 70 % de la demanda estimada
MIN_SERVICE_RATIO = 0.20  ### Cada producto debe cubrir al menos el 20 % de su demanda
MAX_CONCENTRATION_RATIO = 0.35  ### Ningún producto puede superar el 35 % de la capacidad total

# Cálculo de la demanda y capacidad disponibles
## Se convierten los porcentajes establecidos en límites enteros de unidades
demanda_total = int(parametros["demanda"].sum())  ### Suma la demanda estimada de los 10 productos
capacidad_total = int(np.floor(CAPACITY_RATIO * demanda_total))  ### Calcula la capacidad total disponible redondeando hacia abajo
limite_concentracion = int(np.floor(MAX_CONCENTRATION_RATIO * capacidad_total))  ### Calcula el máximo de unidades asignables a un SKU

# Cálculo del servicio mínimo por producto
## Se garantiza una asignación mínima proporcional a la demanda de cada SKU
parametros["min_servicio"] = np.maximum(
    np.ceil(MIN_SERVICE_RATIO * parametros["demanda"]).astype(int), 1  ### Redondea hacia arriba el 20 % de la demanda y garantiza al menos una unidad
)

# Validación de factibilidad del modelo
## Se verifica que los mínimos de servicio respeten la capacidad y el límite por producto
assert parametros["min_servicio"].sum() <= capacidad_total  ### Comprueba que la suma de mínimos no exceda la capacidad total
assert (parametros["min_servicio"] <= limite_concentracion).all()  ### Comprueba que ningún mínimo supere el límite de concentración por SKU

# Presentación de los parámetros calculados
## Se muestran la demanda total, la capacidad disponible y los parámetros de cada producto
print("Demanda:", demanda_total, "· capacidad:", capacidad_total)  ### Muestra las unidades demandadas y la capacidad total disponible
display(parametros)  ### Presenta la tabla actualizada con el servicio mínimo de cada SKU

Demanda: 9982 · capacidad: 6987


,demanda,price,weeks_present,demand_total,min_servicio
StockCode,,,,,
22197,1809,0.85,12,21703,362
84077,1223,0.29,12,14676,245
85099B,1175,2.08,12,14098,235
22086,1080,2.95,12,12951,216
23084,959,2.08,12,11505,192
84879,860,1.69,12,10310,172
22952,765,0.55,12,9177,153
22577,746,0.29,12,8948,150
22578,693,0.29,12,8305,139


### **Definición de la capacidad y el servicio mínimo**

A partir de una demanda semanal estimada de **9,982 unidades**, se estableció una capacidad disponible de **6,987 unidades**, equivalente al 70 % de la demanda total. Además, se calculó un servicio mínimo del 20 % de la demanda de cada producto, garantizando una asignación inicial que respete las necesidades relativas de cada SKU. Estos límites son supuestos académicos del caso y servirán como restricciones para optimizar la asignación de inventario.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Demanda semanal total | 9,982 unidades | Demanda estimada conjunta de los 10 productos. |
| Capacidad disponible | 6,987 unidades | Inventario máximo disponible para asignar. |
| Capacidad respecto a la demanda | 70 % | Proporción de demanda que puede cubrirse como máximo. |
| Servicio mínimo | 20 % por SKU | Cobertura mínima exigida para cada producto. |
| Concentración máxima | 35 % de la capacidad | Límite individual de asignación por producto, equivalente a 2,445 unidades. |
| Factibilidad inicial | Cumple las validaciones | Los mínimos de servicio caben en la capacidad y respetan el límite individual. |

> **Importante:** 70 % de capacidad, 20 % de servicio mínimo y 35 % de concentración son supuestos académicos del caso. No provienen de UCI.

### **Pregunta 3**
**Separe parámetros derivados de UCI y supuestos de operaciones.**

Los parámetros derivados de UCI provienen de los datos históricos de ventas del dataset Online Retail II, mientras que los supuestos de operaciones son decisiones académicas establecidas para formular el modelo de optimización de inventario.
Parámetros derivados de UCI	Supuestos de operaciones
Demanda semanal estimada por SKU.	Capacidad disponible equivalente al 70 % de la demanda total.
Precio unitario mediano de cada producto.	Servicio mínimo del 20 % de la demanda por SKU.
Demanda acumulada durante las 12 semanas analizadas.	Concentración máxima del 35 % de la capacidad para cada SKU.
Frecuencia de ventas por producto, medida en semanas con ventas observadas.	Selección de los 10 productos principales y reglas de asignación del inventario.
Periodo histórico de ventas: del 12/09/2011 al 04/12/2011.	Objetivo de maximizar el valor potencial de las ventas sujeto a las restricciones definidas.

En síntesis: UCI proporciona la evidencia histórica para estimar la demanda y los precios; los supuestos operativos representan las condiciones de capacidad, servicio y concentración bajo las cuales se busca la asignación óptima.

# 5. Línea base intuitiva: asignación proporcional a la demanda

In [8]:
# ══════════════════════════════════════════════════════════════
# LÍNEA BASE INTUITIVA DE ASIGNACIÓN PROPORCIONAL
# ══════════════════════════════════════════════════════════════

# Definición de la función de asignación proporcional
## Se distribuye la capacidad según la demanda, respetando los límites operativos
def proportional_allocation(params, capacity, min_col="min_servicio", max_conc=None):
    """Regla proporcional con mínimos, demanda y concentración: baseline factible."""

    # Definición del límite superior por producto
    ## La asignación no puede superar la demanda ni el tope de concentración establecido
    upper = params["demanda"].astype(int).copy()  ### Establece la demanda como límite superior de cada SKU
    if max_conc is not None:
        upper = upper.clip(upper=int(max_conc))  ### Aplica el límite máximo de concentración cuando está definido

    # Definición del límite inferior por producto
    ## Cada SKU debe recibir al menos el servicio mínimo requerido
    lower = params[min_col].astype(int).copy()  ### Obtiene la asignación mínima permitida por producto

    # Validación de factibilidad de la política
    ## Se comprueba que los límites permitan distribuir toda la capacidad
    if (lower > upper).any() or lower.sum() > capacity or upper.sum() < capacity:
        raise ValueError("La política no permite asignar toda la capacidad.")  ### Detiene la ejecución si la asignación no es factible

    # Asignación inicial y cálculo de unidades restantes
    ## Se parte de los mínimos y se distribuye la capacidad restante
    alloc = lower.copy()  ### Inicializa la asignación con el servicio mínimo de cada SKU
    remaining = int(capacity - alloc.sum())  ### Calcula las unidades disponibles después de cubrir los mínimos
    weights = params["demanda"].astype(float)  ### Usa la demanda como peso para repartir las unidades restantes

    # Distribución iterativa de la capacidad restante
    ## Se asignan unidades proporcionalmente, respetando los límites de cada producto
    while remaining:
        room = upper - alloc  ### Calcula cuántas unidades adicionales puede recibir cada SKU
        active = room > 0  ### Identifica los productos que todavía pueden recibir unidades
        if not active.any():
            break  ### Finaliza si ningún producto tiene capacidad adicional disponible

        quota = remaining * weights[active] / weights[active].sum()  ### Calcula la cuota proporcional de cada SKU activo
        add = np.minimum(np.floor(quota).astype(int), room[active])  ### Asigna unidades enteras sin superar los límites disponibles

        if add.sum() == 0:
            order = (quota - np.floor(quota)).sort_values(ascending=False).index  ### Ordena los SKU por la parte decimal de su cuota
            for sku in order:
                if remaining == 0: break  ### Detiene el reparto cuando ya no quedan unidades
                if room.loc[sku] > 0:
                    alloc.loc[sku] += 1  ### Asigna una unidad al SKU con mayor resto decimal
                    remaining -= 1  ### Reduce la capacidad restante en una unidad
        else:
            alloc.loc[active] += add  ### Incorpora las unidades calculadas a los SKU activos
            remaining -= int(add.sum())  ### Actualiza la capacidad restante después de la asignación

    # Validación de la asignación final
    ## Se verifica que se haya distribuido exactamente toda la capacidad
    assert alloc.sum() == capacity  ### Confirma que las unidades asignadas coincidan con la capacidad disponible
    return alloc  ### Devuelve la asignación final por producto

# Aplicación de la línea base y cálculo del valor potencial
## Se utiliza la regla proporcional como referencia para comparar la solución óptima
baseline = parametros.copy()  ### Copia los parámetros originales para conservarlos sin modificaciones
baseline["asignacion_intuitiva"] = proportional_allocation(
    parametros, capacidad_total, max_conc=limite_concentracion
)  ### Calcula la asignación proporcional respetando las restricciones
baseline["valor_intuitivo"] = baseline["asignacion_intuitiva"] * baseline["price"]  ### Estima el valor potencial por SKU según su precio mediano

# Presentación de resultados de la línea base
## Se muestran las unidades asignadas y su valor potencial total
display(baseline)  ### Presenta los parámetros y resultados de la política intuitiva
print("Unidades intuitivas:", int(baseline["asignacion_intuitiva"].sum()))  ### Muestra el total de unidades asignadas
print("Valor potencial intuitivo (£):", round(baseline["valor_intuitivo"].sum(), 2))  ### Muestra el valor potencial total en libras esterlinas

,demanda,price,weeks_present,demand_total,min_servicio,asignacion_intuitiva,valor_intuitivo
StockCode,,,,,,,
22197,1809,0.85,12,21703,362,1266,1076.10
84077,1223,0.29,12,14676,245,857,248.53
85099B,1175,2.08,12,14098,235,823,1711.84
22086,1080,2.95,12,12951,216,756,2230.20
23084,959,2.08,12,11505,192,672,1397.76
84879,860,1.69,12,10310,172,601,1015.69
22952,765,0.55,12,9177,153,535,294.25
22577,746,0.29,12,8948,150,522,151.38
22578,693,0.29,12,8305,139,485,140.65


Unidades intuitivas: 6987
Valor potencial intuitivo (£): 8853.9


### **Evaluación de la línea base de asignación intuitiva**

La regla proporcional distribuyó las **6,987 unidades disponibles** entre los 10 productos, respetando los mínimos de servicio, la demanda estimada y el límite de concentración por SKU. El valor potencial de ventas obtenido fue de **£8,853.90**, que servirá como referencia para evaluar la mejora de la solución óptima. Esta política distribuye el inventario según la demanda, sin priorizar directamente los productos que generan mayor valor económico por unidad.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Capacidad asignada | 6,987 unidades | Se utilizó el 100 % de la capacidad disponible. |
| Productos evaluados | 10 SKU | Se incluyeron los 10 productos seleccionados. |
| Valor potencial de ventas | £8,853.90 | Valor estimado según las unidades asignadas y los precios medianos. |
| Método de asignación | Proporcional a la demanda | Distribuye las unidades según la demanda de cada producto. |
| Función en el análisis | Línea base | Permite comparar el desempeño económico del modelo de optimización. |

### **Pregunta 4**
**¿Por qué la comparación con PuLP sería engañosa si la regla intuitiva excediera un límite?**

La comparación con PuLP sería engañosa si la regla intuitiva excediera algún límite porque ambos métodos deben cumplir las mismas restricciones operativas. Si la regla intuitiva asignara más unidades que la capacidad disponible, superara la demanda de un producto o incumpliera el límite de concentración, su valor potencial no sería una alternativa válida. Por tanto, la comparación sería injusta, ya que PuLP estaría resolviendo un problema con restricciones que la regla intuitiva no respeta. Para evaluar correctamente la mejora, ambas soluciones deben ser factibles bajo las mismas condiciones.

# **6. Formulación con PuLP**

In [9]:
# ══════════════════════════════════════════════════════════════
# FORMULACIÓN DEL MODELO DE OPTIMIZACIÓN CON PuLP
# ══════════════════════════════════════════════════════════════

# Definición del problema de optimización
## Se busca maximizar el valor potencial de las ventas sujeto a las restricciones operativas
model = pl.LpProblem("Asignacion_Inventario", pl.LpMaximize)  ### Define un problema de maximización

# Creación de las variables de decisión
## Cada variable representa las unidades enteras asignadas a un producto
x = {sku: pl.LpVariable(f"x_{i}", lowBound=0, cat=pl.LpInteger)
     for i, sku in enumerate(parametros.index)}  ### Crea una variable entera no negativa por cada SKU

# Definición de la función objetivo
## Se maximiza el valor económico de las unidades asignadas según el precio mediano
model += pl.lpSum(parametros.loc[sku, "price"] * x[sku] for sku in parametros.index)  ### Suma el precio por las unidades asignadas a cada producto

# Restricción de capacidad total
## La asignación conjunta no puede superar la capacidad disponible
model += pl.lpSum(x.values()) <= capacidad_total  ### Limita el inventario asignado a la capacidad total

# Restricciones individuales por producto
## Cada SKU debe respetar su demanda, servicio mínimo y límite de concentración
for sku in parametros.index:
    model += x[sku] <= int(parametros.loc[sku, "demanda"])  ### Evita asignar más unidades que la demanda estimada
    model += x[sku] >= int(parametros.loc[sku, "min_servicio"])  ### Garantiza el servicio mínimo establecido
    model += x[sku] <= limite_concentracion  ### Limita la asignación individual según la concentración máxima

# Verificación de la estructura del modelo
## Se comprueba el número de variables y restricciones formuladas
print("Variables:", len(x), "· restricciones:", len(model.constraints))  ### Muestra el tamaño del modelo matemático

Variables: 10 · restricciones: 31


### **Verificación de la formulación del modelo de optimización**

El modelo matemático se formuló correctamente con **10 variables de decisión y 31 restricciones**. Cada variable representa las unidades que se asignarán a uno de los 10 productos. Las restricciones incluyen una condición de capacidad total y tres condiciones por producto: respetar la demanda, garantizar el servicio mínimo y cumplir el límite de concentración. Esta estructura permite buscar la asignación que maximice el valor potencial de ventas dentro de las condiciones operativas establecidas.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Variables de decisión | 10 | Una variable de asignación por cada SKU. |
| Restricciones de capacidad | 1 | Limita la asignación total a la capacidad disponible. |
| Restricciones por SKU | 30 | Tres restricciones para cada uno de los 10 productos. |
| Total de restricciones | 31 | Una restricción global más 30 individuales. |
| Objetivo | Maximizar el valor potencial | Prioriza la asignación según el precio unitario de cada producto. |

### **Pregunta 5**
**Explique por qué maximizar precio × unidades sin restricciones no responde la decisión de negocio.**

Maximizar únicamente el precio por las unidades asignadas, sin restricciones, no responde a la decisión de negocio porque el modelo podría concentrar todo el inventario en el producto de mayor precio, ignorando la capacidad disponible, la demanda real de cada SKU y el nivel mínimo de servicio requerido para los demás productos. Esto produciría una asignación poco realista y podría generar exceso de inventario en algunos productos y desabastecimiento en otros. Por ello, las restricciones permiten que la solución de PuLP maximice el valor potencial de ventas respetando las condiciones operativas del negocio.

# **7. Resolver el problema**

In [10]:
# ══════════════════════════════════════════════════════════════
# EJECUCIÓN DEL MODELO DE OPTIMIZACIÓN CON CBC
# ══════════════════════════════════════════════════════════════

# Configuración y ejecución del solucionador
## Se utiliza CBC, incluido en PuLP, para encontrar la asignación óptima
solver = pl.PULP_CBC_CMD(msg=False)  ### Configura CBC sin mostrar mensajes detallados durante la resolución
status = model.solve(solver)  ### Ejecuta el modelo y almacena el estado de la solución

# Verificación del estado y del valor objetivo
## Se comprueba el estado antes de interpretar el resultado económico
print("Estado:", pl.LpStatus[status])  ### Muestra si la solución es óptima, infactible, no acotada o no resuelta
print("Valor objetivo (£):", round(pl.value(model.objective), 2))  ### Muestra el valor potencial máximo calculado en libras esterlinas

Estado: Optimal
Valor objetivo (£): 11478.03


### **Solución óptima del modelo de inventario**

El solucionador CBC obtuvo el estado **Optimal**, lo que indica que encontró la asignación que maximiza el valor potencial de ventas dentro de las restricciones definidas. El valor objetivo alcanzado es de **£11,478.03**, superior a la línea base intuitiva de £8,853.90. Esto demuestra que la optimización matemática puede mejorar la asignación de inventario al priorizar productos con mayor precio unitario, respetando la capacidad, la demanda y los niveles mínimos de servicio.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Estado del modelo | Optimal | Se encontró una solución óptima para el modelo formulado. |
| Valor potencial óptimo | £11,478.03 | Máximo valor potencial obtenido bajo las restricciones establecidas. |
| Valor de la línea base intuitiva | £8,853.90 | Valor potencial de la asignación proporcional a la demanda. |
| Mejora absoluta | £2,624.13 | Diferencia entre la solución óptima y la línea base. |
| Mejora relativa | 29.64 % | Incremento del valor potencial respecto a la línea base. |

In [11]:
# ══════════════════════════════════════════════════════════════
# EXTRACCIÓN Y VALIDACIÓN DE LA ASIGNACIÓN ÓPTIMA
# ══════════════════════════════════════════════════════════════

# Validación del estado del solucionador
## Se detiene el proceso si CBC no encontró una solución óptima
if pl.LpStatus[status] != "Optimal":
    raise RuntimeError(
        "El modelo no alcanzó estado Optimal. Revise capacidad, mínimos y concentración."
    )

# Registro de la asignación óptima por producto
## Se copia la tabla de parámetros y se incorporan las unidades asignadas por el modelo
opt = parametros.copy()  ### Conserva los parámetros originales para el análisis
opt["asignacion_optima"] = [int(round(x[sku].value())) for sku in opt.index]  ### Extrae las unidades óptimas asignadas a cada SKU
opt["valor_optimo"] = opt["asignacion_optima"] * opt["price"]  ### Calcula el valor potencial de cada producto

# Presentación y verificación de los resultados
## Se muestran las asignaciones y se comprueba el uso de la capacidad disponible
display(opt)  ### Presenta la tabla con los resultados de optimización
print("Unidades asignadas:", int(opt["asignacion_optima"].sum()))  ### Muestra el total de unidades asignadas
print("Valor potencial óptimo (£):", round(opt["valor_optimo"].sum(), 2))  ### Muestra el valor potencial total en libras esterlinas

,demanda,price,weeks_present,demand_total,min_servicio,asignacion_optima,valor_optimo
StockCode,,,,,,,
22197,1809,0.85,12,21703,362,1554,1320.90
84077,1223,0.29,12,14676,245,245,71.05
85099B,1175,2.08,12,14098,235,1175,2444.00
22086,1080,2.95,12,12951,216,1080,3186.00
23084,959,2.08,12,11505,192,959,1994.72
84879,860,1.69,12,10310,172,860,1453.40
22952,765,0.55,12,9177,153,153,84.15
22577,746,0.29,12,8948,150,150,43.50
22578,693,0.29,12,8305,139,139,40.31


Unidades asignadas: 6987
Valor potencial óptimo (£): 11478.03


### **Distribución óptima del inventario por producto**

La solución óptima asignó las **6,987 unidades disponibles** y alcanzó un valor potencial de ventas de **£11,478.03**. El modelo priorizó los productos con mayor precio unitario, como los SKU 22086 (£2.95) y 85099B (£2.08), asignándoles una cantidad mayor, sin superar su demanda estimada. Al mismo tiempo, respetó los mínimos de servicio de los 10 productos y el límite de concentración individual. Esta distribución permite aprovechar mejor la capacidad disponible que la regla proporcional intuitiva.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Unidades asignadas | 6,987 | Se utilizó toda la capacidad disponible. |
| Valor potencial óptimo | £11,478.03 | Valor potencial máximo obtenido por el modelo. |
| Producto con mayor asignación | SKU 22086 | Recibió 1,080 unidades, equivalente a su demanda estimada. |
| Productos atendidos en su demanda total | 4 SKU | Los SKU 85099B, 22086, 23084 y 84879 recibieron el 100 % de su demanda estimada. |
| Restricciones operativas | Cumplidas | La asignación respeta la capacidad, los mínimos de servicio, la demanda y la concentración máxima. |
| Diferencia frente a la línea base | £2,624.13 | Valor potencial adicional respecto a la asignación intuitiva. |
| Mejora relativa | 29.64 % | Incremento del valor potencial frente a la línea base de £8,853.90. |

### **Pregunta 6**
**¿Qué garantiza el estado Optimal y qué no garantiza respecto de la demanda y los costos?**

El estado Optimal garantiza que PuLP encontró la mejor asignación posible para maximizar el valor potencial de ventas, cumpliendo todas las restricciones definidas en el modelo, como la capacidad disponible, la demanda estimada, el servicio mínimo y la concentración máxima por producto.

Sin embargo, no garantiza que la demanda estimada coincida con la demanda real futura, porque se basa en datos históricos y supuestos académicos. Tampoco garantiza la rentabilidad real ni la minimización de costos, ya que la función objetivo considera el precio por las unidades asignadas, pero no incluye costos de compra, almacenamiento, transporte ni pérdidas por productos no vendidos.

# 8. Comparación: intuitivo vs óptimo

In [12]:
# ══════════════════════════════════════════════════════════════
# COMPARACIÓN ENTRE LA ASIGNACIÓN INTUITIVA Y LA ÓPTIMA
# ══════════════════════════════════════════════════════════════

# Consolidación de parámetros y asignaciones por producto
## Se reúnen la demanda, el precio y las unidades asignadas por ambos métodos
comparison = parametros[["demanda","price"]].copy()  ### Copia la demanda estimada y el precio de cada SKU
comparison["intuitivo"] = baseline["asignacion_intuitiva"]  ### Incorpora las unidades asignadas por la regla proporcional
comparison["optimo"] = opt["asignacion_optima"]  ### Incorpora las unidades asignadas por el modelo de PuLP

# Cálculo de diferencias entre métodos
## Se identifican los cambios en unidades y en el valor potencial por producto
comparison["dif_unidades"] = comparison["optimo"] - comparison["intuitivo"]  ### Un valor positivo indica más unidades en la solución óptima
comparison["valor_intuitivo"] = comparison["intuitivo"] * comparison["price"]  ### Calcula el valor potencial de la asignación intuitiva
comparison["valor_optimo"] = comparison["optimo"] * comparison["price"]  ### Calcula el valor potencial de la asignación óptima

# Ordenamiento y presentación de la comparación
## Se muestran primero los productos que ganaron más unidades con la optimización
display(comparison.sort_values("dif_unidades", ascending=False))  ### Ordena los SKU según la diferencia de unidades entre ambos métodos

,demanda,price,intuitivo,optimo,dif_unidades,valor_intuitivo,valor_optimo
StockCode,,,,,,,
85099B,1175,2.08,823,1175,352,1711.84,2444.00
22086,1080,2.95,756,1080,324,2230.20,3186.00
22197,1809,0.85,1266,1554,288,1076.10,1320.90
23084,959,2.08,672,959,287,1397.76,1994.72
84879,860,1.69,601,860,259,1015.69,1453.40
21915,672,1.25,470,672,202,587.50,840.00
22578,693,0.29,485,139,-346,140.65,40.31
22577,746,0.29,522,150,-372,151.38,43.50
22952,765,0.55,535,153,-382,294.25,84.15


### **Comparación de la asignación intuitiva frente a la óptima por producto**

La comparación muestra que el modelo de optimización redistribuye el inventario desde productos de menor precio unitario hacia productos de mayor valor económico. Los SKU 85099B y 22086 reciben más unidades que en la regla intuitiva, mientras que los SKU 84077, 22952, 22577 y 22578 reducen su asignación. Esta redistribución permite elevar el valor potencial total de £8,853.90 a £11,478.03, manteniendo las restricciones operativas definidas.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Valor intuitivo | £8,853.90 | Valor potencial de la asignación proporcional a la demanda. |
| Valor óptimo | £11,478.03 | Valor potencial de la asignación calculada por PuLP. |
| Incremento absoluto | £2,624.13 | Valor potencial adicional obtenido con la optimización. |
| Mejora relativa | 29.64 % | Incremento porcentual respecto a la línea base. |
| Productos con mayor asignación óptima | 6 SKU | Reciben más unidades que con la regla intuitiva. |
| Productos con menor asignación óptima | 4 SKU | Ceden unidades para priorizar productos de mayor precio unitario. |

In [13]:
# ══════════════════════════════════════════════════════════════
# CÁLCULO DE LA MEJORA ECONÓMICA DE LA OPTIMIZACIÓN
# ══════════════════════════════════════════════════════════════

# Cálculo del valor potencial total de cada solución
## Se suman los valores potenciales de todos los productos para ambos métodos
v_int = float(comparison["valor_intuitivo"].sum())  ### Obtiene el valor potencial total de la asignación intuitiva
v_opt = float(comparison["valor_optimo"].sum())  ### Obtiene el valor potencial total de la asignación óptima

# Cálculo de la mejora absoluta y relativa
## Se compara la solución óptima con la línea base intuitiva
improvement = v_opt - v_int  ### Calcula el incremento absoluto del valor potencial en libras esterlinas
improvement_pct = improvement / v_int if v_int else np.nan  ### Calcula la mejora porcentual y evita dividir entre cero

# Presentación de los indicadores económicos
## Se reportan los valores en libras y el porcentaje de mejora
print("Valor intuitivo (£):", round(v_int, 2))  ### Muestra el valor potencial de la línea base
print("Valor óptimo (£):", round(v_opt, 2))  ### Muestra el valor potencial de la solución óptima
print("Mejora absoluta (£):", round(improvement, 2))  ### Muestra la diferencia monetaria entre ambas soluciones
print("Mejora porcentual:", f"{improvement_pct:.2%}")  ### Muestra el incremento relativo respecto a la línea base

Valor intuitivo (£): 8853.9
Valor óptimo (£): 11478.03
Mejora absoluta (£): 2624.13
Mejora porcentual: 29.64%


### **Evaluación de la mejora económica de la optimización**

La solución óptima alcanzó un valor potencial de ventas de **£11,478.03**, frente a los **£8,853.90** de la asignación intuitiva. Esto representa una mejora absoluta de **£2,624.13**, equivalente al **29.64 %** respecto a la línea base. El resultado evidencia que redistribuir el inventario priorizando productos con mayor precio unitario permite incrementar el valor potencial de las ventas bajo las restricciones establecidas. Sin embargo, esta mejora representa ingresos potenciales, no utilidad neta, porque el modelo no considera los costos operativos.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Valor intuitivo | £8,853.90 | Valor potencial de ventas de la asignación proporcional. |
| Valor óptimo | £11,478.03 | Valor potencial de ventas de la asignación calculada por PuLP. |
| Mejora absoluta | £2,624.13 | Incremento del valor potencial frente a la línea base. |
| Mejora porcentual | 29.64 % | Aumento relativo del valor potencial respecto a la asignación intuitiva. |

In [14]:
# ══════════════════════════════════════════════════════════════
# VISUALIZACIÓN COMPARATIVA DE LA ASIGNACIÓN DE INVENTARIO
# ══════════════════════════════════════════════════════════════

# Transformación de los datos al formato largo
## Se organizan las asignaciones para comparar ambos métodos en un mismo gráfico
long = (
    comparison[["intuitivo","optimo"]]  ### Selecciona las asignaciones de ambos métodos
    .reset_index()  ### Convierte StockCode en una columna
    .melt(id_vars="StockCode", var_name="metodo", value_name="unidades")  ### Agrupa los métodos en una columna y sus unidades en otra
)

# Construcción del gráfico de barras agrupadas
## Se comparan las unidades asignadas a cada SKU mediante ambos métodos
fig = px.bar(
    long,
    x="StockCode",  ### Define los productos en el eje horizontal
    y="unidades",  ### Representa las unidades asignadas en el eje vertical
    color="metodo",  ### Diferencia visualmente la regla intuitiva y la solución óptima
    barmode="group",  ### Coloca las barras de ambos métodos lado a lado
    title="Asignación intuitiva vs óptima"  ### Define el título del gráfico
)
fig.show()  ### Muestra el gráfico interactivo

### **Comparación visual de la asignación intuitiva y óptima**

El gráfico muestra cómo PuLP redistribuye las 6,987 unidades disponibles entre los 10 productos. La solución óptima aumenta la asignación de productos con mayor precio unitario, como los SKU 85099B, 22086 y 23084, mientras reduce las unidades destinadas a productos de menor precio, como 84077, 22952, 22577 y 22578. Esta redistribución permite elevar el valor potencial de ventas de £8,853.90 a £11,478.03, lo que representa una mejora del 29.64 %, respetando las restricciones del modelo.

### **Pregunta 7**
**¿Qué SKU ganan y pierden unidades, y cuál es la lógica económica de ese cambio?**

La optimización asigna más unidades a los SKU 22197, 85099B, 22086, 23084, 84879 y 21915, mientras que reduce las unidades de los SKU 84077, 22952, 22577 y 22578 respecto a la regla intuitiva.

La lógica económica consiste en priorizar los productos con mayor precio unitario para maximizar el valor potencial de ventas, respetando la capacidad disponible, la demanda estimada, el servicio mínimo y el límite de concentración. Por ejemplo, el SKU 22086, con un precio de £2.95, recibe toda su demanda estimada de 1,080 unidades; en cambio, el SKU 84077, con un precio de £0.29, recibe únicamente el mínimo de servicio de 245 unidades. Esta redistribución permite mejorar el valor potencial de ventas en un 29.64 % frente a la asignación intuitiva.

# **9. Verificación de restricciones**

In [15]:
# ══════════════════════════════════════════════════════════════
# VERIFICACIÓN DE HOLGURAS Y RESTRICCIONES DEL MODELO
# ══════════════════════════════════════════════════════════════

# Cálculo de holguras por producto
## La holgura representa la diferencia entre el límite establecido y la asignación realizada
checks = pd.DataFrame(index=parametros.index)  ### Crea una tabla para verificar las restricciones de cada SKU
checks["x"] = opt["asignacion_optima"]  ### Registra las unidades asignadas por la solución óptima
checks["demanda"] = parametros["demanda"]  ### Registra la demanda máxima estimada por producto
checks["min_servicio"] = parametros["min_servicio"]  ### Registra el servicio mínimo exigido por producto
checks["holgura_demanda"] = checks["demanda"] - checks["x"]  ### Calcula las unidades de demanda que no se cubrieron
checks["holgura_minimo"] = checks["x"] - checks["min_servicio"]  ### Calcula cuánto supera la asignación al mínimo de servicio
checks["holgura_concentracion"] = limite_concentracion - checks["x"]  ### Calcula cuánto queda antes de alcanzar el límite por SKU
display(checks)  ### Muestra las holguras y asignaciones de cada producto

# Validación de las restricciones
## Se comprueba que no se exceda la capacidad y que ninguna holgura sea negativa
assert checks["x"].sum() <= capacidad_total  ### Verifica que la asignación total no supere la capacidad disponible
assert (checks[["holgura_demanda","holgura_minimo","holgura_concentracion"]] >= 0).all().all()  ### Confirma que se respeten los límites individuales

# Resumen de uso de capacidad y concentración
## Se identifica si algún producto alcanzó exactamente el límite de concentración
print("Capacidad usada:", int(checks["x"].sum()), "/", capacidad_total)  ### Muestra las unidades utilizadas frente a la capacidad total
print("Límite de concentración vinculante:", bool((checks["holgura_concentracion"] == 0).any()))  ### Indica si algún SKU alcanzó el límite máximo permitido

,x,demanda,min_servicio,holgura_demanda,holgura_minimo,holgura_concentracion
StockCode,,,,,,
22197,1554,1809,362,255,1192,891
84077,245,1223,245,978,0,2200
85099B,1175,1175,235,0,940,1270
22086,1080,1080,216,0,864,1365
23084,959,959,192,0,767,1486
84879,860,860,172,0,688,1585
22952,153,765,153,612,0,2292
22577,150,746,150,596,0,2295
22578,139,693,139,554,0,2306


Capacidad usada: 6987 / 6987
Límite de concentración vinculante: False


### **Verificación de holguras y cumplimiento de restricciones**

La solución óptima asignó las **6,987 unidades disponibles**, utilizando el 100 % de la capacidad. Las holguras muestran que los SKU 85099B, 22086, 23084, 84879 y 21915 alcanzaron su demanda estimada, mientras que los SKU 84077, 22952, 22577 y 22578 recibieron únicamente el servicio mínimo. Ningún producto alcanzó el límite máximo de concentración de 2,445 unidades, por lo que esta restricción no fue vinculante en la solución obtenida. Las validaciones confirman que se respetaron los límites establecidos.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Capacidad utilizada | 6,987 de 6,987 unidades | Se utilizó toda la capacidad disponible. |
| Restricción de capacidad | Activa | No quedan unidades disponibles para asignar. |
| Productos que alcanzaron su demanda | 5 SKU | Su holgura de demanda es cero. |
| Productos en el servicio mínimo | 5 SKU | Su holgura del mínimo es cero. |
| Límite de concentración | No vinculante | Ningún SKU alcanzó el máximo permitido de 2,445 unidades. |
| Cumplimiento de restricciones | Correcto | No se detectaron holguras negativas ni exceso de capacidad. |

### **Pregunta 8**
**¿La concentración del 35 % es vinculante en el escenario base? Sustente con las holguras.**

No, la restricción de concentración del 35 % no es vinculante en el escenario base. La capacidad total es de 6,987 unidades, por lo que el límite por SKU es de 2,445 unidades. Sin embargo, la mayor asignación corresponde al SKU 22197, con 1,554 unidades y una holgura de 891 unidades respecto al límite. Además, todos los demás productos presentan holguras positivas, por lo que ninguno alcanza la concentración máxima permitida.

# **10. Sensibilidad a la capacidad**

In [16]:
# ══════════════════════════════════════════════════════════════
# ANÁLISIS DE ESCENARIOS DE CAPACIDAD Y RESTRICCIONES
# ══════════════════════════════════════════════════════════════

# Definición de la función para resolver escenarios operativos
## Se recalculan la capacidad, el servicio mínimo y la concentración para cada escenario
def solve_scenario(params, capacity_ratio=0.70, min_service_ratio=0.20,
                   max_conc_ratio=0.35):
    # Cálculo de capacidad y límite de concentración
    ## Se establecen los límites según los porcentajes definidos para el escenario
    total = int(params["demanda"].sum())  ### Calcula la demanda total de los productos
    capacity = int(np.floor(capacity_ratio * total))  ### Calcula la capacidad disponible redondeando hacia abajo
    max_conc = int(np.floor(max_conc_ratio * capacity))  ### Calcula el máximo de unidades permitidas por SKU
    p = params.copy()  ### Copia los parámetros originales para conservarlos sin modificaciones

    # Cálculo del servicio mínimo del escenario
    ## Se garantiza al menos una unidad por producto
    p["min_scn"] = np.maximum(np.ceil(min_service_ratio * p["demanda"]).astype(int), 1)  ### Calcula el mínimo de servicio según la demanda y el porcentaje establecido

    # Validación de factibilidad
    ## Se comprueba que los mínimos puedan cubrirse con la capacidad y respeten el límite por SKU
    if p["min_scn"].sum() > capacity or (p["min_scn"] > max_conc).any():
        return {"status":"Infeasible", "capacity":capacity, "optimal_value":np.nan,
                "baseline_value":np.nan, "uplift_pct":np.nan, "allocation":None}  ### Devuelve un escenario infactible sin ejecutar el modelo

    # Formulación del modelo entero para el escenario
    ## Se conserva la misma función objetivo y se actualizan las restricciones operativas
    m = pl.LpProblem("Escenario", pl.LpMaximize)  ### Define el problema de maximización del escenario
    z = {sku: pl.LpVariable(f"z_{i}", lowBound=0, cat=pl.LpInteger)
         for i,sku in enumerate(p.index)}  ### Crea una variable entera no negativa por cada SKU
    m += pl.lpSum(p.loc[sku,"price"]*z[sku] for sku in p.index)  ### Maximiza el valor potencial de ventas
    m += pl.lpSum(z.values()) <= capacity  ### Limita la asignación total a la capacidad del escenario
    for sku in p.index:
        m += z[sku] <= int(p.loc[sku,"demanda"])  ### Evita superar la demanda estimada por SKU
        m += z[sku] >= int(p.loc[sku,"min_scn"])  ### Garantiza el servicio mínimo del escenario
        m += z[sku] <= max_conc  ### Aplica el límite de concentración individual

    # Resolución y validación del estado
    ## Se comprueba el estado del solucionador antes de utilizar los resultados
    status = pl.LpStatus[m.solve(pl.PULP_CBC_CMD(msg=False))]  ### Ejecuta CBC y obtiene el estado de la solución
    if status != "Optimal":
        return {"status":status,"capacity":capacity,"optimal_value":np.nan,
                "baseline_value":np.nan,"uplift_pct":np.nan,"allocation":None}  ### Devuelve el estado sin calcular indicadores si no hay óptimo

    # Cálculo de la asignación óptima y la línea base
    ## Ambos métodos utilizan las mismas restricciones para permitir una comparación justa
    allocation = pd.Series({sku:int(round(z[sku].value())) for sku in p.index})  ### Extrae las unidades óptimas asignadas por producto
    baseline_scn = proportional_allocation(p, capacity, min_col="min_scn", max_conc=max_conc)  ### Calcula la asignación proporcional bajo las mismas reglas
    optimal_value = float((allocation*p["price"]).sum())  ### Calcula el valor potencial de la solución óptima
    baseline_value = float((baseline_scn*p["price"]).sum())  ### Calcula el valor potencial de la línea base
    return {"status":status,"capacity":capacity,"optimal_value":optimal_value,
            "baseline_value":baseline_value,
            "uplift_pct":100*(optimal_value/baseline_value-1),
            "allocation":allocation.to_dict()}  ### Devuelve el estado, los valores y la mejora porcentual

# Evaluación de escenarios de capacidad
## Se comparan capacidades equivalentes al 60 %, 70 % y 85 % de la demanda estimada
scenarios = [solve_scenario(parametros, capacity_ratio=r) for r in [0.60,0.70,0.85]]  ### Resuelve los tres escenarios de capacidad

# Consolidación de los resultados
## Se construye una tabla con los principales indicadores de cada escenario
scenario_df = pd.DataFrame([
    {"capacity_ratio": ratio, **{k:v for k,v in result.items() if k != "allocation"}}
    for ratio, result in zip([0.60,0.70,0.85], scenarios)
])  ### Combina el porcentaje de capacidad con los resultados obtenidos

# Preparación del indicador porcentual para la visualización
## Se convierte la mejora de porcentaje entero a proporción decimal para el formato porcentual de Plotly
scenario_df["improvement_pct"] = scenario_df["uplift_pct"] / 100  ### Convierte, por ejemplo, 40.07 en 0.4007 para mostrar 40.07 %
display(scenario_df)  ### Muestra los resultados comparativos de los escenarios

,capacity_ratio,status,capacity,optimal_value,baseline_value,uplift_pct,improvement_pct
0,0.60,Optimal,5989,10629.73,7588.60,40.074981,0.400750
1,0.70,Optimal,6987,11478.03,8853.90,29.638126,0.296381
2,0.85,Optimal,8484,12214.08,10751.27,13.605927,0.136059


### **Análisis de sensibilidad según la capacidad disponible**

Los tres escenarios alcanzaron el estado **Optimal**, lo que indica que se encontró una solución óptima bajo las restricciones de cada escenario. Al aumentar la capacidad del 60 % al 85 % de la demanda, el valor potencial óptimo crece de £10,629.73 a £12,214.08. Sin embargo, la mejora porcentual frente a la línea base disminuye del 40.07 % al 13.61 %, porque al disponer de más capacidad, la asignación proporcional también puede cubrir una mayor parte de la demanda. Esto muestra que una capacidad más limitada hace más importante optimizar la distribución del inventario.

| Indicador | Capacidad 60 % | Capacidad 70 % | Capacidad 85 % |
|---|---:|---:|---:|
| Estado | Optimal | Optimal | Optimal |
| Capacidad disponible | 5,989 | 6,987 | 8,484 |
| Valor óptimo | £10,629.73 | £11,478.03 | £12,214.08 |
| Valor de línea base | £7,588.60 | £8,853.90 | £10,751.27 |
| Mejora absoluta | £3,041.13 | £2,624.13 | £1,462.81 |
| Mejora porcentual | 40.07 % | 29.64 % | 13.61 % |

In [17]:
# ══════════════════════════════════════════════════════════════
# VISUALIZACIÓN DE LA MEJORA SEGÚN LA CAPACIDAD DISPONIBLE
# ══════════════════════════════════════════════════════════════

# Construcción del gráfico de barras
## Se compara la mejora porcentual de la optimización para cada escenario de capacidad
fig = px.bar(
    scenario_df,
    x="capacity_ratio",  ### Representa la capacidad disponible como proporción de la demanda
    y="improvement_pct",  ### Muestra la mejora porcentual frente a la asignación intuitiva
    text=scenario_df["improvement_pct"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "NA"),  ### Etiqueta cada barra con su porcentaje de mejora
    title="Ventaja de optimizar según capacidad disponible",  ### Define el título del gráfico
    labels={"capacity_ratio":"Capacidad / demanda", "improvement_pct":"Mejora vs intuitivo"}  ### Establece los nombres de los ejes
)
fig.show()  ### Muestra el gráfico interactivo de comparación

### **Impacto de la capacidad disponible en la ventaja de optimizar**

El gráfico muestra una relación inversa entre la capacidad disponible y la mejora porcentual obtenida mediante la optimización. Con una capacidad del 60 % de la demanda, la solución óptima supera en **40.1 %** a la asignación intuitiva; con el 70 %, la mejora es del **29.6 %**; y con el 85 %, disminuye al **13.6 %**. Esto ocurre porque, cuando el inventario es más limitado, distribuirlo estratégicamente entre los productos de mayor valor económico genera una ventaja más significativa frente a la regla proporcional. Al aumentar la capacidad, ambos métodos pueden atender una mayor proporción de la demanda y la diferencia relativa se reduce.

### **Pregunta 9**
**¿Por qué la mejora porcentual de PuLP cambia cuando la capacidad pasa de 60 % a 85 %?**

La mejora porcentual de PuLP cambia porque la capacidad disponible determina qué tan importante es distribuir estratégicamente el inventario. Con una capacidad del 60 %, la optimización supera en 40.07 % a la regla intuitiva, ya que priorizar los productos de mayor precio genera una ventaja considerable cuando el inventario es limitado. En cambio, con una capacidad del 85 %, la mejora disminuye al 13.61 %, porque ambos métodos pueden atender una mayor proporción de la demanda y sus resultados económicos se acercan.

Por tanto, cuanto más limitada es la capacidad, mayor puede ser la ventaja relativa de optimizar, aunque el valor potencial absoluto de la solución óptima aumente cuando se dispone de más inventario.

# 11. Decisión y recomendación

### **Pregunta 10**
**¿Recomendaría operar con la asignación PuLP? Indique una condición que podría cambiar la respuesta.**

Sí, recomendaría operar con la asignación de PuLP como propuesta inicial para la toma de decisiones, porque incrementa el valor potencial de ventas en un 29.64 %, pasando de £8,853.90 a £11,478.03, sin exceder la capacidad disponible ni incumplir las restricciones operativas.

Sin embargo, antes de implementarla, validaría los resultados con datos de demanda más recientes y los costos reales de compra, almacenamiento y transporte. Una condición que cambiaría mi recomendación sería que la asignación óptima generara mayores costos operativos o pérdidas por productos no vendidos, reduciendo o eliminando la rentabilidad esperada.

# 12. Retos individuales de aplicación

La parte guiada ya formuló y verificó el modelo. Resuelva los tres retos en su notebook personal. Son **10 preguntas** en total (4 + 3 + 3). En cada respuesta distinga resultados calculados, supuestos y recomendación. No altere las transacciones para producir una mejora mayor.

## Reto 1 — ¿Conviene ampliar la capacidad? (4 preguntas)

**Caso.** Operaciones puede aumentar la capacidad semanal, pero debe pagar por ella. Compare 60 %, 70 % y 85 % de la demanda total. Mantenga las demás reglas.

**R1.1.** Construya una tabla con capacidad, valor intuitivo, valor PuLP y mejora porcentual para los tres escenarios. ¿Qué representa cada cifra?

**R1.2.** Calcule el valor marginal por unidad adicional de capacidad entre 60 %→70 % y 70 %→85 %. ¿Por qué no son iguales?

**R1.3.** Si una unidad adicional de capacidad cuesta £0,65 por semana, ¿cuál ampliación conviene respecto del escenario de 60 %? Muestre valor incremental, costo incremental y saldo neto.

**R1.4.** Recomiende a operaciones una capacidad e indique una limitación de su cálculo.

In [18]:
# ══════════════════════════════════════════════════════════════
# RETO 1: ¿CONVIENE AMPLIAR LA CAPACIDAD?
# ══════════════════════════════════════════════════════════════

# R1.1. Comparación de los tres escenarios
## Se presentan la capacidad, los valores potenciales y la mejora porcentual
tabla_r1 = scenario_df[[
    "capacity_ratio", "capacity", "baseline_value",
    "optimal_value", "uplift_pct"
]].copy()

tabla_r1 = tabla_r1.rename(columns={
    "capacity_ratio": "capacidad_pct",
    "capacity": "capacidad_unidades",
    "baseline_value": "valor_intuitivo",
    "optimal_value": "valor_pulp",
    "uplift_pct": "mejora_pct"
})

tabla_r1["mejora_pct"] = tabla_r1["mejora_pct"].round(2)
display(tabla_r1)

# R1.2. Valor marginal de ampliar la capacidad
## Se calcula el incremento del valor óptimo por cada unidad adicional de capacidad
escenarios_ordenados = scenario_df.sort_values("capacity_ratio").reset_index(drop=True)

marginal = []
for i in range(1, len(escenarios_ordenados)):
    anterior = escenarios_ordenados.iloc[i - 1]
    actual = escenarios_ordenados.iloc[i]

    unidades_adicionales = int(actual["capacity"] - anterior["capacity"])
    valor_incremental = actual["optimal_value"] - anterior["optimal_value"]

    marginal.append({
        "comparacion": f'{anterior["capacity_ratio"]:.0%} → {actual["capacity_ratio"]:.0%}',
        "unidades_adicionales": unidades_adicionales,
        "valor_incremental": round(valor_incremental, 2),
        "valor_marginal_por_unidad": round(valor_incremental / unidades_adicionales, 4)
    })

tabla_marginal = pd.DataFrame(marginal)
display(tabla_marginal)

# R1.3. Evaluación del costo de capacidad adicional
## Se compara cada ampliación con el escenario inicial del 60 %
COSTO_UNIDAD = 0.65  ### Costo supuesto en libras por unidad adicional de capacidad semanal

base = escenarios_ordenados.iloc[0]
evaluacion = []

for i in range(1, len(escenarios_ordenados)):
    actual = escenarios_ordenados.iloc[i]

    unidades_adicionales = int(actual["capacity"] - base["capacity"])
    valor_incremental = actual["optimal_value"] - base["optimal_value"]
    costo_incremental = unidades_adicionales * COSTO_UNIDAD

    evaluacion.append({
        "escenario": f'{actual["capacity_ratio"]:.0%}',
        "unidades_adicionales": unidades_adicionales,
        "valor_incremental": round(valor_incremental, 2),
        "costo_incremental": round(costo_incremental, 2),
        "saldo_neto": round(valor_incremental - costo_incremental, 2)
    })

tabla_costos = pd.DataFrame(evaluacion)
display(tabla_costos)

# R1.4. Identificación de la alternativa con mayor saldo neto
## Se selecciona la ampliación con mayor saldo neto frente al escenario del 60 %
mejor = tabla_costos.loc[tabla_costos["saldo_neto"].idxmax()]

print("Ampliación recomendada:", mejor["escenario"])
print("Saldo neto incremental (£):", mejor["saldo_neto"])

,capacidad_pct,capacidad_unidades,valor_intuitivo,valor_pulp,mejora_pct
0,0.60,5989,7588.60,10629.73,40.07
1,0.70,6987,8853.90,11478.03,29.64
2,0.85,8484,10751.27,12214.08,13.61


,comparacion,unidades_adicionales,valor_incremental,valor_marginal_por_unidad
0,60% → 70%,998,848.30,0.8500
1,70% → 85%,1497,736.05,0.4917


,escenario,unidades_adicionales,valor_incremental,costo_incremental,saldo_neto
0,70%,998,848.30,648.70,199.6
1,85%,2495,1584.35,1621.75,-37.4


Ampliación recomendada: 70%
Saldo neto incremental (£): 199.6


### **Reto 1.1. Comparación de escenarios de capacidad**

Los resultados muestran que aumentar la capacidad incrementa el valor potencial de ventas de la solución óptima, desde £10,629.73 con una capacidad del 60 % hasta £12,214.08 con el 85 %. Sin embargo, la mejora porcentual frente a la asignación intuitiva disminuye del 40.07 % al 13.61 %, porque ambos métodos pueden atender una mayor demanda al disponer de más inventario.

| Indicador | Capacidad 60 % | Capacidad 70 % | Capacidad 85 % |
|---|---:|---:|---:|
| Capacidad disponible | 5,989 | 6,987 | 8,484 |
| Valor intuitivo | £7,588.60 | £8,853.90 | £10,751.27 |
| Valor PuLP | £10,629.73 | £11,478.03 | £12,214.08 |
| Mejora porcentual | 40.07 % | 29.64 % | 13.61 % |

### **Reto 1.2. Valor marginal de la capacidad adicional**

El valor marginal disminuye al ampliar la capacidad. Entre el 60 % y el 70 %, cada unidad adicional genera aproximadamente £0.85 de valor potencial; entre el 70 % y el 85 %, genera alrededor de £0.49. Esto ocurre porque las unidades adicionales disponibles pueden destinarse a productos con menor precio unitario o menor oportunidad de venta.

| Ampliación | Unidades adicionales | Valor incremental | Valor marginal por unidad |
|---|---:|---:|---:|
| 60 % → 70 % | 998 | £848.30 | £0.8500 |
| 70 % → 85 % | 1,497 | £736.05 | £0.4917 |

### **Reto 1.3. Evaluación del costo de ampliar la capacidad**

Con un costo supuesto de £0.65 por unidad adicional de capacidad semanal, conviene ampliar del 60 % al 70 %, porque el valor incremental supera el costo en £199.60. En cambio, ampliar del 60 % al 85 % genera un saldo neto negativo de £37.40.

| Ampliación desde 60 % | Valor incremental | Costo incremental | Saldo neto |
|---|---:|---:|---:|
| 60 % → 70 % | £848.30 | £648.70 | **+£199.60** |
| 60 % → 85 % | £1,584.35 | £1,621.75 | **−£37.40** |

### **Reto 1.4. Recomendación a operaciones**

Recomendaría una capacidad del **70 % de la demanda estimada**, equivalente a 6,987 unidades semanales. Bajo el costo supuesto, esta ampliación ofrece un saldo neto incremental positivo de £199.60 respecto al escenario del 60 %, mientras que ampliar al 85 % no resulta conveniente.

**Limitación:** el análisis compara valor potencial de ventas con costo de capacidad, pero no incorpora costos de compra, almacenamiento, transporte ni incertidumbre de la demanda. Antes de implementar la decisión, se deben validar los márgenes reales y los costos operativos.

## Reto 2 — Servicio mínimo y costo de la equidad (3 preguntas)

**Caso.** Comercial propone elevar el servicio mínimo para evitar productos casi desabastecidos. Evalúe 20 %, 30 % y 50 % con capacidad 70 %.

**R2.1.** Resuelva los tres escenarios y muestre valor óptimo y unidades por SKU. ¿Cuál sigue siendo factible?

**R2.2.** Calcule cuánto valor potencial se sacrifica al pasar de 20 % a 30 % y de 20 % a 50 %. Identifique qué SKU cambian más.

**R2.3.** Recomiende una política de servicio. Explique qué indicador operativo faltaría para decidirla en una empresa real.

In [19]:

# ══════════════════════════════════════════════════════════════
# RETO 2: SERVICIO MÍNIMO Y COSTO DE LA EQUIDAD
# ══════════════════════════════════════════════════════════════

# R2.1. Resolución de escenarios con capacidad fija del 70 %
## Se comparan servicios mínimos del 20 %, 30 % y 50 %
service_ratios = [0.20, 0.30, 0.50]
service_results = {
    ratio: solve_scenario(
        parametros,
        capacity_ratio=0.70,
        min_service_ratio=ratio
    )
    for ratio in service_ratios
}

# Resumen de factibilidad y valor potencial
## Se registra el estado, la capacidad y el valor óptimo de cada escenario
service_summary = pd.DataFrame([
    {
        "servicio_minimo": ratio,
        "estado": result["status"],
        "capacidad": result["capacity"],
        "valor_optimo": result["optimal_value"]
    }
    for ratio, result in service_results.items()
])
display(service_summary)

# Comparación de asignaciones por SKU
## Se reúnen las unidades asignadas en cada escenario factible
allocation_comparison = pd.DataFrame(index=parametros.index)
allocation_comparison["demanda"] = parametros["demanda"]

for ratio, result in service_results.items():
    column = f"asignacion_{int(ratio * 100)}pct"
    allocation_comparison[column] = (
        pd.Series(result["allocation"])
        if result["allocation"] is not None
        else np.nan
    )

display(allocation_comparison)

# R2.2. Cálculo del valor potencial sacrificado
## Se toma el escenario del 20 % como referencia económica
base_value = service_results[0.20]["optimal_value"]

sacrifice_rows = []
for ratio in service_ratios:
    result = service_results[ratio]
    sacrifice_rows.append({
        "servicio_minimo": ratio,
        "valor_optimo": result["optimal_value"],
        "valor_sacrificado": base_value - result["optimal_value"]
    })

sacrifice_df = pd.DataFrame(sacrifice_rows)
sacrifice_df["sacrificio_pct"] = (
    sacrifice_df["valor_sacrificado"] / base_value * 100
)
display(sacrifice_df)

# Diferencias de asignación respecto al servicio del 20 %
## Se identifican los SKU con mayores cambios en unidades
for ratio in [0.30, 0.50]:
    col = f"asignacion_{int(ratio * 100)}pct"
    base_col = "asignacion_20pct"
    allocation_comparison[f"cambio_vs_{int(ratio * 100)}pct"] = (
        allocation_comparison[col] - allocation_comparison[base_col]
    )

display(allocation_comparison)


,servicio_minimo,estado,capacidad,valor_optimo
0,0.2,Optimal,6987,11478.03
1,0.3,Optimal,6987,11306.53
2,0.5,Optimal,6987,10791.63


,demanda,asignacion_20pct,asignacion_30pct,asignacion_50pct
StockCode,,,,
22197,1809,1554,1212,905
84077,1223,245,367,612
85099B,1175,1175,1175,1175
22086,1080,1080,1080,1080
23084,959,959,959,959
84879,860,860,860,817
22952,765,153,230,383
22577,746,150,224,373
22578,693,139,208,347


,servicio_minimo,valor_optimo,valor_sacrificado,sacrificio_pct
0,0.2,11478.03,0.0,0.000000
1,0.3,11306.53,171.5,1.494159
2,0.5,10791.63,686.4,5.980120


,demanda,asignacion_20pct,asignacion_30pct,asignacion_50pct,cambio_vs_30pct,cambio_vs_50pct
StockCode,,,,,,
22197,1809,1554,1212,905,-342,-649
84077,1223,245,367,612,122,367
85099B,1175,1175,1175,1175,0,0
22086,1080,1080,1080,1080,0,0
23084,959,959,959,959,0,0
84879,860,860,860,817,0,-43
22952,765,153,230,383,77,230
22577,746,150,224,373,74,223
22578,693,139,208,347,69,208


### **R2.1. Factibilidad y asignación según el servicio mínimo**

Los tres escenarios son factibles y mantienen una capacidad de 6,987 unidades semanales. Al elevar el servicio mínimo del 20 % al 50 %, el modelo redistribuye el inventario para garantizar una cobertura mayor de los productos con menor asignación inicial. Los SKU 85099B, 22086 y 23084 mantienen su demanda completamente cubierta, mientras que el SKU 22197 reduce su asignación de 1,554 a 905 unidades para liberar inventario destinado a otros productos.

| Servicio mínimo | Estado | Capacidad | Valor óptimo |
|---|---|---:|---:|
| 20 % | Optimal | 6,987 | £11,478.03 |
| 30 % | Optimal | 6,987 | £11,306.53 |
| 50 % | Optimal | 6,987 | £10,791.63 |

### **R2.2. Costo económico de elevar el servicio mínimo**

El valor potencial disminuye al exigir una cobertura mínima mayor. Frente al escenario del 20 %, establecer un mínimo del 30 % sacrifica £171.50, mientras que elevarlo al 50 % sacrifica £686.40. El cambio más significativo ocurre en el SKU 22197, cuya asignación disminuye en 342 unidades al pasar al 30 % y en 649 unidades al pasar al 50 %. Los SKU 84077, 22952, 22577 y 22578 reciben más unidades para cumplir los mínimos de servicio.

| Comparación | Valor potencial sacrificado | Sacrificio porcentual |
|---|---:|---:|
| 20 % → 30 % | £171.50 | 1.49 % |
| 20 % → 50 % | £686.40 | 5.98 % |

### **R2.3. Recomendación de política de servicio**

Recomendaría inicialmente un servicio mínimo del **30 %**, porque mejora la cobertura exigida con una reducción relativamente moderada del valor potencial, equivalente al 1.49 % frente al escenario del 20 %. El mínimo del 50 % podría considerarse si la empresa necesita reducir significativamente el riesgo de desabastecimiento de productos de menor asignación.

En una operación real faltaría medir el **nivel de servicio observado**, como la tasa de cumplimiento de pedidos (*fill rate*), además de la frecuencia de quiebres de stock. Estos indicadores permitirían comprobar si la mayor cobertura mejora efectivamente la atención al cliente y justifica el sacrificio económico.

## Reto 3 — ¿La concentración limita de verdad? (3 preguntas)

**Caso.** Riesgos propone limitar cada SKU al 20 %, 25 %, 35 % o 50 % de la capacidad. Mantenga el escenario de capacidad 70 % y servicio mínimo 20 %.

**R3.1.** Resuelva las cuatro políticas. Presente valor óptimo, asignación y factibilidad.

**R3.2.** Indique en qué políticas la restricción cambia la solución y calcule el valor perdido frente a 35 %. ¿Qué demuestra el escenario de 50 %?

**R3.3.** Recomiende un límite de concentración y explique por qué el porcentaje por sí solo no mide todo el riesgo de inventario.

In [20]:

# ══════════════════════════════════════════════════════════════
# RETO 3: ANÁLISIS DE SENSIBILIDAD DE LA CONCENTRACIÓN
# ══════════════════════════════════════════════════════════════

# R3.1. Resolución de las cuatro políticas de concentración
## Se mantienen la capacidad del 70 % y el servicio mínimo del 20 %
concentration_ratios = [0.20, 0.25, 0.35, 0.50]

concentration_results = {
    ratio: solve_scenario(
        parametros,
        capacity_ratio=0.70,
        min_service_ratio=0.20,
        max_conc_ratio=ratio
    )
    for ratio in concentration_ratios
}

# Resumen de factibilidad y valor óptimo
## Se registra el límite individual en unidades y el valor potencial de cada política
concentration_summary = pd.DataFrame([
    {
        "concentracion_pct": ratio,
        "limite_sku": int(np.floor(ratio * result["capacity"])),
        "estado": result["status"],
        "capacidad": result["capacity"],
        "valor_optimo": result["optimal_value"]
    }
    for ratio, result in concentration_results.items()
])

display(concentration_summary)

# Comparación de asignaciones por SKU
## Se organiza la asignación de cada política en columnas separadas
concentration_allocation = pd.DataFrame(index=parametros.index)
concentration_allocation["demanda"] = parametros["demanda"]
concentration_allocation["price"] = parametros["price"]

for ratio, result in concentration_results.items():
    column = f"asignacion_{int(ratio * 100)}pct"
    concentration_allocation[column] = (
        pd.Series(result["allocation"])
        if result["allocation"] is not None
        else np.nan
    )

display(concentration_allocation)

# R3.2. Valor perdido frente a la política del 35 %
## Se utiliza el escenario del 35 % como referencia
reference_value = concentration_results[0.35]["optimal_value"]

loss_rows = []
for ratio, result in concentration_results.items():
    loss = reference_value - result["optimal_value"]
    loss_rows.append({
        "concentracion_pct": ratio,
        "estado": result["status"],
        "valor_optimo": result["optimal_value"],
        "valor_perdido_vs_35pct": loss,
        "perdida_pct_vs_35pct": (
            100 * loss / reference_value
            if pd.notna(reference_value) and reference_value != 0
            else np.nan
        )
    })

concentration_loss = pd.DataFrame(loss_rows)
display(concentration_loss)

# Cambios de asignación frente al escenario del 35 %
## Se identifican los SKU que reciben o pierden unidades bajo cada política
reference_column = "asignacion_35pct"

for ratio in [0.20, 0.25, 0.50]:
    column = f"asignacion_{int(ratio * 100)}pct"
    concentration_allocation[f"cambio_{int(ratio * 100)}_vs_35"] = (
        concentration_allocation[column]
        - concentration_allocation[reference_column]
    )

display(concentration_allocation)


,concentracion_pct,limite_sku,estado,capacidad,valor_optimo
0,0.20,1397,Optimal,6987,11430.93
1,0.25,1746,Optimal,6987,11478.03
2,0.35,2445,Optimal,6987,11478.03
3,0.50,3493,Optimal,6987,11478.03


,demanda,price,asignacion_20pct,asignacion_25pct,asignacion_35pct,asignacion_50pct
StockCode,,,,,,
22197,1809,0.85,1397,1554,1554,1554
84077,1223,0.29,245,245,245,245
85099B,1175,2.08,1175,1175,1175,1175
22086,1080,2.95,1080,1080,1080,1080
23084,959,2.08,959,959,959,959
84879,860,1.69,860,860,860,860
22952,765,0.55,310,153,153,153
22577,746,0.29,150,150,150,150
22578,693,0.29,139,139,139,139


,concentracion_pct,estado,valor_optimo,valor_perdido_vs_35pct,perdida_pct_vs_35pct
0,0.20,Optimal,11430.93,47.1,0.410349
1,0.25,Optimal,11478.03,0.0,0.000000
2,0.35,Optimal,11478.03,0.0,0.000000
3,0.50,Optimal,11478.03,0.0,0.000000


,demanda,price,asignacion_20pct,asignacion_25pct,asignacion_35pct,asignacion_50pct,cambio_20_vs_35,cambio_25_vs_35,cambio_50_vs_35
StockCode,,,,,,,,,
22197,1809,0.85,1397,1554,1554,1554,-157,0,0
84077,1223,0.29,245,245,245,245,0,0,0
85099B,1175,2.08,1175,1175,1175,1175,0,0,0
22086,1080,2.95,1080,1080,1080,1080,0,0,0
23084,959,2.08,959,959,959,959,0,0,0
84879,860,1.69,860,860,860,860,0,0,0
22952,765,0.55,310,153,153,153,157,0,0
22577,746,0.29,150,150,150,150,0,0,0
22578,693,0.29,139,139,139,139,0,0,0


### **R3.1. Evaluación de las políticas de concentración**

Los cuatro escenarios son factibles y presentan un estado óptimo. Con un límite de concentración del 20 %, el valor potencial es de £11,430.93. Al elevar el límite al 25 %, alcanza £11,478.03, valor que se mantiene en los escenarios del 35 % y 50 %. Esto indica que una política del 25 % es suficiente para alcanzar el máximo valor potencial con las restricciones actuales.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Capacidad disponible | 6,987 unidades | Límite total de inventario asignable. |
| Valor óptimo con concentración del 20 % | £11,430.93 | Menor valor potencial de los escenarios evaluados. |
| Valor óptimo con concentración del 25 % | £11,478.03 | Alcanza el máximo valor potencial. |
| Estado de las soluciones | Optimal en los cuatro escenarios | Todas las políticas tienen una solución óptima. |

### **R3.2. Impacto de las políticas de concentración**

El límite del 20 % es el único que modifica la asignación respecto al escenario de referencia del 35 %. El producto 22197 recibe 157 unidades menos, mientras que el producto 22952 recibe 157 unidades adicionales. Esta redistribución reduce el valor potencial en £47.10, equivalente al 0.41 %. Los límites del 25 % y 50 % mantienen la misma asignación que el 35 %, sin generar pérdidas ni beneficios adicionales.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Pérdida con concentración del 20 % | £47.10 (0.41 %) | La restricción más estricta reduce ligeramente el valor potencial. |
| Pérdida con concentración del 25 % | £0.00 (0 %) | No hay pérdida respecto al escenario del 35 %. |
| Pérdida con concentración del 50 % | £0.00 (0 %) | Permitir mayor concentración no mejora el resultado. |
| Cambio en el producto 22197 con concentración del 20 % | -157 unidades | Se reduce su asignación. |
| Cambio en el producto 22952 con concentración del 20 % | +157 unidades | Recibe las unidades redistribuidas. |

### **R3.3. Recomendación de la política de concentración**

Se recomienda un límite de concentración del **25 %**, porque permite alcanzar el valor potencial máximo de £11,478.03 sin necesidad de flexibilizar la política hasta el 35 % o el 50 %. El límite del 20 % también es viable, pero genera una pérdida de £47.10. La recomendación debe validarse con información real sobre rotación, variabilidad de la demanda y costos de faltantes.

El porcentaje de concentración no representa por sí solo todo el riesgo del inventario. También deben considerarse los tiempos de reposición, la variabilidad de la demanda, la importancia de cada producto, la obsolescencia y el costo de quedarse sin existencias.

# 13. Verificaciones finales

- Revise que cada solución factible respete capacidad, demanda, servicio y concentración.
- Indique si la regla intuitiva usa exactamente las mismas restricciones.
- Compruebe que las diez preguntas de los retos tengan respuesta numérica o una decisión justificada.
- Declare una limitación: ventas observadas ≠ demanda total; valor potencial ≠ utilidad.

# 14. Cierre ejecutivo y ticket de salida

Se recomienda trabajar con una capacidad del 70 %, equivalente a 6,987 unidades, porque ofrece un equilibrio entre capacidad y valor potencial.  
La regla intuitiva genera £8,853.90, mientras que PuLP alcanza £11,478.03.  
La optimización mejora el valor potencial en £2,624.13, equivalente al 29.64 %.  
Ampliar la capacidad del 60 % al 70 % genera £848.30 adicionales de valor potencial, frente a un costo estimado de £648.70, con un beneficio neto de £199.60.  
Se recomienda un servicio mínimo del 30 % como alternativa intermedia, considerando que su valor óptimo es £11,306.53.  
Para la concentración, se recomienda un límite del 25 %, pues alcanza el máximo valor potencial sin necesidad de flexibilizarlo más.  
El principal riesgo es que la demanda histórica no represente la demanda futura y que el valor potencial no equivalga a la utilidad real.  
La recomendación cambiaría si los costos reales, la variabilidad de la demanda o los requisitos de servicio justificaran otra asignación.

**Ticket:** «Con 6,987 unidades de capacidad, PuLP genera £11,478.03 frente a £8,853.90 de la regla intuitiva. Recomiendo una capacidad del 70 %, un servicio mínimo del 30 % y una concentración máxima del 25 %, porque equilibran el valor potencial, el servicio y la diversificación. Esta conclusión cambiaría si los costos reales, la demanda futura o los niveles de servicio exigidos modificaran la rentabilidad de la solución».

# 15. Entrega y referencias

Suba el notebook ejecutado a `semana11/` de su repositorio personal y comparta el enlace de Colab. Incluya los tres retos y las diez respuestas individuales. Declare el uso de IA.

Chen, D. (2012). *Online Retail II* [dataset]. UCI Machine Learning Repository. DOI: 10.24432/C5CG6D.
Sharda, R., Delen, D. y Turban, E. *Business Intelligence, Analytics, Data Science and AI*.
Winston, W. L. *Operations Research: Applications and Algorithms*.